# Leiturista x CRISP-DM — brainstorm

**Data:** 2026-08-16 | **Disciplina:** Projeto 4 - DADOS (Cesar School, BD2026.2)
**Grupo:** 3 | **Cliente:** distribuidora de energia elétrica
**Repo:** `jhlr/leiturista`

**Referência:** Wirth, R. & Hipp, J. (2000). *CRISP-DM: Towards a Standard Process Model for
Data Mining*. O método da disciplina já é declarado como "ciclos inspirados no CRISP-DM"
(`docs/projeto4_desafio.md`). O que este notebook faz é levar o leiturista de "inspiração"
para execução explícita do método — identificando onde cada fase agrega valor real.


## 0. Estado atual do leiturista (para ancorar o brainstorm)

| Fase CRISP-DM   | Onde o leiturista está hoje |
|-----------------|-----------------------------|
| Business Understanding | Critério de sucesso não fechado com o cliente |
| Data Understanding | UFPR-AMR público + ~3.840 imgs baixadas; lote real só no Kickoff (12/09) |
| Data Preparation | Pipeline det (PP-OCRv5_mobile) + rec (PP-OCRv6_tiny / TrOCR-small) |
| Modeling        | Baselines off-the-shelf + fine-tune TrOCR-stage1 (MLflow em `mlflow.db`) |
| Evaluation      | Benchmark no test UFPR-AMR: ~25-36% leituras exatas, 77-85% por dígito |
| Deployment      | Demo Streamlit + Release `modelos-1.0` |

A execução dos ciclos abaixo é o que fecha as lacunas de **business**, **dado real** e **manutenção**.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch

phases = [
    ("1. Business\nUnderstanding", "gap"),
    ("2. Data\nUnderstanding", "parcial"),
    ("3. Data\nPreparation", "ok"),
    ("4. Modeling", "ok"),
    ("5. Evaluation", "ok"),
    ("6. Deployment", "parcial"),
]
colors = {
    "ok": "#2e7d32",
    "parcial": "#ef6c00",
    "gap": "#c62828",
}
n = len(phases)
R, r = 1.0, 0.52
fig, ax = plt.subplots(figsize=(9, 9), dpi=110)
wedges = []
for i, (label, status) in enumerate(phases):
    theta0 = 90 - i * 360 / n
    theta1 = 90 - (i + 1) * 360 / n
    w = plt.matplotlib.patches.Wedge((0, 0), R, theta0, theta1, width=R - r,
                                     facecolor=colors[status], edgecolor="white", lw=2)
    ax.add_patch(w)
    mid = np.radians((theta0 + theta1) / 2)
    x, y = (R + r) / 2 * np.cos(mid), (R + r) / 2 * np.sin(mid)
    ax.text(x, y, label, ha="center", va="center", fontsize=10, color="white", fontweight="bold")
ax.text(0, 0.12, "CRISP-DM", ha="center", fontsize=22, fontweight="bold")
ax.text(0, -0.12, "ciclo iterativo", ha="center", fontsize=11, color="#555")

ax.annotate("", xy=(R + 0.12, 0), xytext=(0.05, R + 0.12),
            arrowprops=dict(arrowstyle="-|>", color="#333", lw=1.6, connectionstyle="arc3,rad=0.55"))
ax.text(0.42, R + 0.05, "deploy retorna\ndados reais", fontsize=8.5, color="#333")

legend = [plt.matplotlib.patches.Patch(color=c, label=t) for t, c in
          [("ativa no leiturista", "#2e7d32"), ("parcial / em espera", "#ef6c00"),
           ("gap que o CRISP-DM expoe", "#c62828")]]
ax.legend(handles=legend, loc="upper left", bbox_to_anchor=(0.98, 0.62), fontsize=9)
ax.set_xlim(-1.45, 1.45); ax.set_ylim(-1.35, 1.35); ax.axis("off")
plt.title("Leiturista no ciclo CRISP-DM: onde estamos, onde falta", fontsize=13)
plt.tight_layout()
plt.show()

## 1. O ciclo CRISP-DM aplicado ao leiturista

Cores por fase: **verde** = já ativa, **laranja** = parcial/em espera, **vermelho** = lacuna que o
método expõe. O deploy da demo já aponta de volta para o início do ciclo — que é exatamente o
retorno de dados que a Tarefa 2 precisa.


## 2. Business Understanding — o maior ganho em aberto

O leiturista já vive em *Modeling* (baselines PP-OCRv6, fine-tune TrOCR-stage1) e tocou
*Deployment* (demo Streamlit, MLflow, Release `modelos-1.0`), mas a fase que decide o valor do
negócio está subvalorizada: **não há critério de sucesso fechado com a distribuidora**. Qual a taxa
de erro aceitável — por dígito ou por leitura exata? E, decisivo, quanto custa um **falso aceite**
(leitura/billing errado passando) versus uma **revisão manual** (revisita do leiturista)? Sem esses
limiares, os "77-85% por dígito" do benchmark não dizem se o sistema está bom. O CRISP-DM força
escrever esses critérios *antes* de otimizar modelo — e isso vira contrato mensurável para a
Tarefa 1 e, mais ainda, **desbloqueia a Tarefa 2**: "coerência foto x ocorrência" só vira tarefa
de ML quando o cliente define formalmente o que é coerente (portão fechado para I100? medidor do
cliente certo?) — vira uma lista de casos aceitáveis/rejeitáveis de anotar.


## 3. Fechar o ciclo com a distribuição real (retorno do deploy)

UFPR-AMR é público e ótimo, mas é outra distribuição: ângulo, luz, serial que vaza como "serial",
odômetro pequeno não segmentado pelo detector — o relatório de benchmark já flagra esses limites.
O CRISP-DM trata o **lote real da distribuidora** (Kickoff em 12/09) como *Data Understanding /
Data Preparation / Evaluation de volta*: cada foto real com a ocorrência do leiturista é, ao mesmo
tempo, avaliação da Tarefa 1 na distribuição-alvo **e** rótulo para a Tarefa 2. O mesmo dado que
fecha o GAP da validação serve para medir o quanto o TrOCR-stage1 generaliza — esse é exatamente o
retorno que o ciclo CRISP-DM desenha, e o leiturista já tem a infra para registrá-lo (um
experimento por task no MLflow, predições como blobs, docs datados).


## 4. Deployment continuo e governanca

A demo Streamlit já é um piloto de campo, mas CRISP-DM pede **monitoramento** e critério explícito
de *quando* re-treinar (câmera do leiturista muda, medidores mudam, iluminação muda). A regra do
grupo — doc datado commitado em `docs/` — é o próprio artefato de governança que o método exige:
cada ciclo vira algo reproduzível (`run_id` no `mlflow.db`) em vez de conversa. Ganho líquido:
passar de "ciclos inspirados no CRISP-DM" para execução explícita do método que a disciplina já cobra.
